In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization MiddleWare

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [4]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a39c6453-c502-4c32-9fb8-e916e559ee03'), AIMessage(content="<think>\nOkay, so I need to figure out what 2 + 2 is. Let me start by recalling basic addition. When you add two numbers together, you're combining their quantities. So 2 plus 2 would be like having two apples and then getting two more apples. How many apples do you have in total? Let's count them: 1, 2, then 3, 4. That makes four apples. \n\nWait, maybe I should visualize it. If I draw two lines: | | and then another two lines: | |, putting them together would be | | | |, which is four lines. Hmm, that seems right. \n\nAlternatively, using fingers. If I hold up two fingers on one hand and two on the other, that's four fingers total. Yeah, that checks out. \n\nI remember learning addition tables as a kid. The 2s row goes 2+0=2, 2+1=3, 2+2=4, so that's in the table. \n\nBut wait, is there any context where 2+2 might not b

### token Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [6]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~130 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='77865893-fa4c-4f29-8450-a1e465ed8fef'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Paris. Let me check the tools available. There's a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters should be a JSON object with the city name. I'll structure the tool call accordingly.\n", 'tool_calls': [{'id': 'p0svfr8sf', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 155, 'total_tokens': 249, 'completion_time': 0.159312869, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006489744, 'prompt_tokens_details': None, 'queue_time': 0.04986504, 'total_time': 0.165802613}, 'model_

### fraction
trigger=("fraction", 0.005),  # 0.5% = ~640 tokens

keep=("fraction", 0.002),     # 0.2% = ~256 tokens

### HUMAN IN THE LOOP

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email ID: {email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"


In [16]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [20]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='249ec49a-8ad8-4de0-b52b-15d056097be3'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, body is How are you? So I need to call send_email_tool with those arguments. No issues here. Just structure the JSON with those parameters.\n", 'tool_calls': [{'id': 'gxpb05j3y', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completio

In [21]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print(" Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f" Result: {result['messages'][-1].content}")

 Paused! Approving...
 Result: ✅ Email with subject "Hello" has been successfully sent to john@test.com. Let me know if you need to send or read any other emails!


Reject

In [22]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [23]:
if "__interrupt__" in result:
    print("Paused ")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    print("Result",result["messages"][-1].content)

Paused 
Result It seems the request to send an email to john@test.com was rejected by the system (rejection ID: 0b8fvj4k9). Would you like me to help you try again, modify the email, or explore other options?


Edit

In [24]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [25]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print(" Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f" Result: {result['messages'][-1].content}")

 Paused! Editing...
 Result: The email has been successfully sent to **correct@email.com** with the subject **"Corrected Subject"**. 

Let me know if you need any further adjustments or additional actions!
